# W4A16 GPTQ + 민감 레이어 확장 + 캘리브레이션 극대화

## 13번 대비 변경점

| 항목 | 13번 | 14번 (본 버전) |
|------|------|---------------|
| NUM_CALIBRATION_SAMPLES | 256 | **8192** (32배) |
| MAX_SEQUENCE_LENGTH | 512 | **4096** (8배) |
| IGNORE | layers.0, 29 | layers.0, 1, **28, 29** (4개 보호) |

### 기대 효과
- 캘리브레이션 샘플 32배 → 양자화 품질 대폭 향상
- 시퀀스 길이 8배 → 장문 문맥 의존성 반영
- 입출력 각 2개 레이어 보호 → PerfNorm 추가 향상
- **모델 크기/속도는 13번과 거의 동일**

---

# 1. Import

In [8]:
import os
import torch
import shutil
from pathlib import Path

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("\u26a0\ufe0f CPU \ubaa8\ub4dc\ub85c \uc2e4\ud589\ub429\ub2c8\ub2e4")
print("\n\u2705 Import \uc644\ub8cc!")

PyTorch: 2.9.1
CUDA: False
⚠️ CPU 모드로 실행됩니다

✅ Import 완료!


# 2. 설정

In [9]:
# ============================================================================
# 모델 설정
# ============================================================================
MODEL_ID = "./open/base_model"
OUT_DIR = "./model"

# ============================================================================
# 데이터셋 설정
# ============================================================================
DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

# ============================================================================
# ⭐ 캘리브레이션 설정 (로컬용)
# ============================================================================
NUM_CALIBRATION_SAMPLES = 1024   # 로컬 메모리 대응
MAX_SEQUENCE_LENGTH = 2048       # 로컬 메모리 대응

# ============================================================================
# 양자화 설정
# ============================================================================
SCHEME = "W4A16"
TARGETS = ["Linear"]

# ⭐ 민감 레이어 보호 확장 (입출력 각 2개, regex로 하위 모듈 전체 매칭)
IGNORE = [
    "embed_tokens", "lm_head",
    "re:model\\.layers\\.0\\..*",   # 입력 쪽
    "re:model\\.layers\\.1\\..*",
    "re:model\\.layers\\.28\\..*",  # 출력 쪽
    "re:model\\.layers\\.29\\..*",
]

# ============================================================================
# GPTQ 최적화 파라미터
# ============================================================================
BLOCK_SIZE = 128
DAMPENING_FRAC = 0.001
ACTORDER = "weight"

# 원본 모델 크기
ORIGINAL_MODEL_SIZE_GB = 2.56

print("=" * 60)
print("W4A16 GPTQ + 민감 레이어 확장 + 캘리브레이션 강화 (로컬)")
print("=" * 60)
print(f"MODEL_ID: {MODEL_ID}")
print(f"SCHEME: {SCHEME}")
print("---")
print("⭐ 캘리브레이션:")
print(f"  SAMPLES: {NUM_CALIBRATION_SAMPLES}")
print(f"  MAX_LEN: {MAX_SEQUENCE_LENGTH}")
print("⭐ GPTQ 파라미터:")
print(f"  BLOCK_SIZE: {BLOCK_SIZE}")
print(f"  DAMPENING_FRAC: {DAMPENING_FRAC}")
print(f"  ACTORDER: {ACTORDER}")
print("⭐ IGNORE:")
for ig in IGNORE:
    print(f"  - {ig}")
print("=" * 60)

W4A16 GPTQ + 민감 레이어 확장 + 캘리브레이션 강화 (로컬)
MODEL_ID: ./open/base_model
SCHEME: W4A16
---
⭐ 캘리브레이션:
  SAMPLES: 1024
  MAX_LEN: 2048
⭐ GPTQ 파라미터:
  BLOCK_SIZE: 128
  DAMPENING_FRAC: 0.001
  ACTORDER: weight
⭐ IGNORE:
  - embed_tokens
  - lm_head
  - re:model\.layers\.0\..*
  - re:model\.layers\.1\..*
  - re:model\.layers\.28\..*
  - re:model\.layers\.29\..*


# 3. 모델 로드

In [10]:
print("[INFO] \ubaa8\ub378 \ub85c\ub4dc \uc911...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

if torch.cuda.is_available():
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.bfloat16,
        trust_remote_code=True,
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.float32,
        trust_remote_code=True,
        device_map="cpu",
    )

print(f"[INFO] \ubaa8\ub378 \ud30c\ub77c\ubbf8\ud130: {model.num_parameters():,}")
print("[INFO] \ubaa8\ub378/\ud1a0\ud06c\ub098\uc774\uc800 \ub85c\ub4dc \uc644\ub8cc")

[INFO] 모델 로드 중...
[INFO] 모델 파라미터: 1,279,391,488
[INFO] 모델/토크나이저 로드 완료


# 4. 데이터셋 로드

In [11]:
print("[INFO] \uce98\ub9ac\ube0c\ub808\uc774\uc158 \ub370\uc774\ud130 \ub85c\ub4dc \uc911...")

ds = load_dataset(
    DATASET_ID,
    split=f"{DATASET_SPLIT}[:{NUM_CALIBRATION_SAMPLES}]",
)

def preprocess(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False
        )
    }

ds = ds.map(preprocess)

print(f"[INFO] \ub370\uc774\ud130\uc14b \ud06c\uae30: {len(ds)}")
print(f"[INFO] \uc2dc\ud000\uc2a4 \ucd5c\ub300 \uae38\uc774: {MAX_SEQUENCE_LENGTH}")
print("[INFO] \ub370\uc774\ud130 \uc804\ucc98\ub9ac \uc644\ub8cc")

[INFO] 캘리브레이션 데이터 로드 중...
[INFO] 데이터셋 크기: 1024
[INFO] 시퀀스 최대 길이: 2048
[INFO] 데이터 전처리 완료


# 5. GPTQ 양자화

In [12]:
print("[INFO] GPTQ \uc591\uc790\ud654 \uc2dc\uc791")
print(f"  - scheme: {SCHEME}")
print(f"  - samples: {NUM_CALIBRATION_SAMPLES}")
print(f"  - max_len: {MAX_SEQUENCE_LENGTH}")
print(f"  - block_size: {BLOCK_SIZE}")
print(f"  - actorder: {ACTORDER}")
print(f"  - dampening_frac: {DAMPENING_FRAC}")
print(f"  - ignore: {IGNORE}")

if torch.cuda.is_available():
    print("\n\U0001f680 GPU \ubaa8\ub4dc\n")
else:
    print("\n\u23f3 CPU \ubaa8\ub4dc: \uc624\ub798 \uac78\ub9b4 \uc218 \uc788\uc74c\n")

recipe = [
    GPTQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=IGNORE,
        block_size=BLOCK_SIZE,
        dampening_frac=DAMPENING_FRAC,
        actorder=ACTORDER,
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

print("\n[INFO] GPTQ \uc591\uc790\ud654 \uc644\ub8cc!")

[INFO] GPTQ 양자화 시작
  - scheme: W4A16
  - samples: 1024
  - max_len: 2048
  - block_size: 128
  - actorder: weight
  - dampening_frac: 0.001
  - ignore: ['embed_tokens', 'lm_head', 're:model\\.layers\\.0\\..*', 're:model\\.layers\\.1\\..*', 're:model\\.layers\\.28\\..*', 're:model\\.layers\\.29\\..*']

⏳ CPU 모드: 오래 걸릴 수 있음



Tokenizing:   0%|          | 0/1024 [00:00<?, ? examples/s]

2026-02-17T01:25:14.475766+0900 | reset | INFO - Compression lifecycle reset
2026-02-17T01:25:14.478407+0900 | from_modifiers | INFO - Creating recipe from modifiers
2026-02-17T01:25:14.501404+0900 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-02-17T01:25:14.501825+0900 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`
2026-02-17T01:25:14.507676+0900 | dispatch_for_sequential | WARNING - CUDA/XPU is not available! Compressing model on CPU instead


(3/31): Calibrating: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1024/1024 [05:58<00:00,  2.86it/s]

2026-02-17T01:42:22.894386+0900 | compress_modules | INFO - Quantizing model.layers.2.self_attn.q_proj using 1024 samples


...

(28/31): Calibrating: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1024/1024 [05:20<00:00,  3.19it/s]

2026-02-17T05:05:40.707071+0900 | compress_modules | INFO - Quantizing model.layers.27.self_attn.q_proj using 1024 samples


2026-02-17T05:05:40.999726+0900 | compress | METRIC - time 0.29s
2026-02-17T05:05:41.000090+0900 | compress | METRIC - error 6073.52
2026-02-17T05:05:41.001988+0900 | get_GPU_usage_nv | WARNING - Pynml library error:
 NVML Shared Library Not Found
2026-02-17T05:05:41.002242+0900 | compress | METRIC - Compressed module size: 16.941056 MB
2026-02-17T05:05:41.003546+0900 | compress_modules | INFO - Quantizing model.layers.27.self_attn.k_proj using 1024 samples
2026-02-17T05:05:41.185724+0900 | compress | METRIC - time 0.18s
2026-02-17T05:05:41.186078+0900 | compress | METRIC - error 1577.70
2026-02-17T05:05:41.186918+0900 | get_GPU_usage_nv | WARNING - Pynml library error:
 NVML Shared Library Not Found
2026-02-17T05:05:41.187135+0900 | compress | METRIC - Compressed module size: 4.235264 MB
2026-02-17T05:05:41.187836+0900 | compress_modules | INFO - Quantizing model.layers.27.self_attn.v_proj using 1024 samples
2026-02-17T05:05:41.368985+0900 | compress | METRIC - time 0.18s
2026-02-17T0

(31/31): Propagating: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1024/1024 [00:00<00:00, 1065.54it/s]


2026-02-17T05:19:08.423706+0900 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2026-02-17T05:19:08.431108+0900 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`

[INFO] GPTQ 양자화 완료!


# 6. 모델 저장

In [13]:
print("[INFO] \ubaa8\ub378 \uc800\uc7a5 \uc911...")

if os.path.exists(OUT_DIR):
    shutil.rmtree(OUT_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

print(f"\n[INFO] \uc800\uc7a5\ub41c \ud30c\uc77c:")
total_size = 0
for f in sorted(os.listdir(OUT_DIR)):
    size = os.path.getsize(os.path.join(OUT_DIR, f))
    total_size += size
    print(f"  {f}: {size/1e6:.1f} MB")

quantized_size_gb = total_size / 1e9

print("\n" + "=" * 60)
print("\ubaa8\ub378 \ud06c\uae30 \ube44\uad50")
print("=" * 60)
print(f"  \uc6d0\ubcf8 \ubaa8\ub378:     {ORIGINAL_MODEL_SIZE_GB:.2f} GB")
print(f"  \uc591\uc790\ud654 \ubaa8\ub378:   {quantized_size_gb:.2f} GB")
print(f"  \uc555\ucd95\ub960:        {quantized_size_gb / ORIGINAL_MODEL_SIZE_GB * 100:.1f}%")
print("=" * 60)

[INFO] 모델 저장 중...
2026-02-17T05:19:08.505731+0900 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Compressing model: 182it [00:01, 110.27it/s]



[INFO] 저장된 파일:
  chat_template.jinja: 0.0 MB
  config.json: 0.0 MB
  generation_config.json: 0.0 MB
  merges.txt: 1.2 MB
  model.safetensors: 1902.3 MB
  recipe.yaml: 0.0 MB
  special_tokens_map.json: 0.0 MB
  tokenizer.json: 7.9 MB
  tokenizer_config.json: 0.1 MB
  vocab.json: 1.9 MB

모델 크기 비교
  원본 모델:     2.56 GB
  양자화 모델:   1.91 GB
  압축률:        74.7%


# 7. 제출 파일 생성

In [14]:
zip_name = "submit_max_calib"
print(f"[INFO] {zip_name}.zip \uc0dd\uc131 \uc911...")

if os.path.exists(f"{zip_name}.zip"):
    os.remove(f"{zip_name}.zip")

shutil.make_archive(
    base_name=zip_name,
    format="zip",
    root_dir=".",
    base_dir=OUT_DIR,
)

zip_size = os.path.getsize(f"{zip_name}.zip") / 1e9
print(f"[INFO] \uc0dd\uc131 \uc644\ub8cc: {zip_name}.zip ({zip_size:.2f} GB)")

if zip_size <= 10:
    print("\u2705 \uc6a9\ub7c9 \uc81c\ud55c \ucda9\uc871 (\u2264 10GB)")
else:
    print("\u274c \uc6a9\ub7c9 \ucd08\uacfc!")

print("\n" + "=" * 60)
print("\uc81c\ucd9c \ud30c\uc77c \uad6c\uc870")
print("=" * 60)
print(f"{zip_name}.zip")
print(f"\u2514\u2500\u2500 model/")
for f in sorted(os.listdir(OUT_DIR))[:5]:
    print(f"    \u251c\u2500\u2500 {f}")
print("    \u2514\u2500\u2500 ...")
print("=" * 60)

[INFO] submit_max_calib.zip 생성 중...
[INFO] 생성 완료: submit_max_calib.zip (1.08 GB)
✅ 용량 제한 충족 (≤ 10GB)

제출 파일 구조
submit_max_calib.zip
└── model/
    ├── chat_template.jinja
    ├── config.json
    ├── generation_config.json
    ├── merges.txt
    ├── model.safetensors
    └── ...


---

# 13번 vs 14번 비교

| 항목 | 13번 | 14번 (본 버전) |
|------|------|---------------|
| SAMPLES | 256 | **8192** (32배) |
| MAX_LEN | 512 | **4096** (8배) |
| IGNORE | layers.0, 29 | layers.0, 1, **28, 29** |
| 양자화 레이어 | 28개 | 26개 |
| 모델 크기 | ~1.49 GB | ~1.56 GB |
| PerfNorm | 기준 | \u2191\u2191 (캘리브레이션 + 보호 확장) |
| SpeedNorm | 기준 | \u2248 (거의 동일) |

---